In [2]:
# KG1 Nemotron - H100 candidate-only one-cell pre-score V7
# Nao faz submit. Roda anchors conhecidos e calibra o gate local contra scores Kaggle reais.

import os, sys, json, shutil, zipfile, subprocess, importlib.util, inspect, runpy
from pathlib import Path

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

# =========================
# Configuracao principal
# =========================
DRIVE_DIR = Path("/content/drive/MyDrive/KG1_Nemotron_prescore")
WORK_DIR = Path("/content/kg1_prescore")
PACK_ZIP = DRIVE_DIR / "kg1_prescore_calibration_pack.zip"
DRIVE_ADAPTERS_ROOT = DRIVE_DIR / "adapters"
LOCAL_ANCHORS_ROOT = Path("/content/kg1_prescore_anchors")
OUT_DIR = DRIVE_DIR / "prescore_outputs_h100_v2"
LOCAL_OFFLOAD_DIR = Path("/content/kg1_prescore_model_offload_h100_v2")

# H100 candidate-only preset. Roda so o candidato mais recente em ~20-45 min.
# Para o passe robusto final, use N_SAMPLES=120, MAX_ADAPTERS=8, MAX_NEW_TOKENS=2048.
N_SAMPLES = 40
MAX_ADAPTERS = 1
MAX_NEW_TOKENS = 1024
MAX_MODEL_LEN = 8192
SCORE_MODE = "raw"
GPU_MEMORY_MARGIN_GB = 8
USE_CACHE = False
FINGERPRINT_MODE = "partial"
DELETE_ANCHOR_ZIPS_AFTER_EXTRACT = True
CANDIDATE_ONLY = True
CANDIDATE_FILTER = ""  # opcional: substring do path do adapter para forcar um candidato especifico
REQUIRE_CALIBRATION_ANCHORS = False
DATASET_RELATIVE = "data/v94/v94_equation_crypt_val.jsonl"

DEFAULT_ANCHORS = {
    "canonical_086": {
        "hf_file": "data/v87_delta/canonical_086_submission.zip",
        "kaggle_score": 0.86,
    },
    "v088_084": {
        "hf_file": "prescore/anchors/v088_084_submission.zip",
        "kaggle_score": 0.84,
    },
}

def sh(cmd: str, check: bool = True):
    print("\n$ " + cmd, flush=True)
    p = subprocess.run(cmd, shell=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")
    return p.returncode

def run_script_in_process(script_path: Path, argv: list[str]):
    old_argv = sys.argv[:]
    try:
        sys.argv = [str(script_path)] + argv
        print("\n$ runpy", " ".join(sys.argv), flush=True)
        try:
            runpy.run_path(str(script_path), run_name="__main__")
        except SystemExit as exc:
            code = exc.code if isinstance(exc.code, int) else 0
            if code:
                raise RuntimeError(f"{script_path} exited with code {code}") from exc
    finally:
        sys.argv = old_argv

print("=== 1. GPU / Python ===", flush=True)
sh("nvidia-smi", check=False)
print(sys.version, flush=True)

print("\n=== 2. Install dependencies ===", flush=True)
sh('pip -q install -U "transformers==5.3.0" "peft==0.18.1" accelerate bitsandbytes safetensors huggingface_hub sentencepiece tqdm kaggle "jedi>=0.16"')
sh('pip -q install -U "torchao>=0.16.0"')
sh("pip -q install -U packaging ninja wheel setuptools")
sh('MAX_JOBS=4 pip -q install -U "causal-conv1d>=1.4.0" "mamba-ssm>=2.2.4" --no-build-isolation')

import transformers, torchao
print("torchao:", getattr(torchao, "__version__", "unknown"), flush=True)
print("transformers:", transformers.__version__, flush=True)
assert hasattr(transformers, "WeightConverter"), "transformers sem WeightConverter; precisa de transformers 5.x"
assert "distributed_operation" in str(inspect.signature(transformers.WeightConverter)), inspect.signature(transformers.WeightConverter)
assert importlib.util.find_spec("mamba_ssm") is not None, "mamba_ssm nao foi instalado"

print("\n=== 3. Mount Drive / secrets ===", flush=True)
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    def secret(name, default=""):
        try:
            return userdata.get(name) or default
        except Exception:
            return default
except Exception:
    userdata = None
    def secret(name, default=""):
        return os.environ.get(name, default)

HF_TOKEN = secret("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
KAGGLE_USERNAME = secret("KAGGLE_USERNAME", os.environ.get("KAGGLE_USERNAME", ""))
KAGGLE_KEY = secret("KAGGLE_KEY", os.environ.get("KAGGLE_KEY", ""))

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY

WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ADAPTERS_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ANCHORS_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR, flush=True)
print("DRIVE_DIR:", DRIVE_DIR, flush=True)
print("DRIVE_ADAPTERS_ROOT:", DRIVE_ADAPTERS_ROOT, flush=True)
print("LOCAL_ANCHORS_ROOT:", LOCAL_ANCHORS_ROOT, flush=True)
print("OUT_DIR:", OUT_DIR, flush=True)

print("\n=== 4. Download/extract latest pre-score pack ===", flush=True)
from huggingface_hub import hf_hub_download
downloaded_pack = hf_hub_download(
    repo_id="felipesp1983/kg1-nemotron-training",
    repo_type="dataset",
    filename="prescore/kg1_prescore_calibration_pack.zip",
    token=HF_TOKEN or None,
    force_download=True,
)
shutil.copy2(downloaded_pack, PACK_ZIP)
print("Pack salvo em:", PACK_ZIP, "size=", PACK_ZIP.stat().st_size, flush=True)

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK_ZIP) as zf:
    zf.extractall(WORK_DIR)
print("Pack extraido.", flush=True)

batch_script = WORK_DIR / "scripts" / "batch_validate_prescore.py"
validate_script = WORK_DIR / "scripts" / "validate_prescore_gate_calibration.py"
assert batch_script.exists(), batch_script
assert validate_script.exists(), validate_script
batch_text = batch_script.read_text(encoding="utf-8")
validate_text = validate_script.read_text(encoding="utf-8")
assert "--fingerprint-mode" in batch_text, "Pack antigo: falta --fingerprint-mode"
assert "--offload-dir" in batch_text, "Pack antigo: falta --offload-dir local"
assert "line_buffering=True" in batch_text, "Pack antigo: falta line buffering"
assert "max_model_len" in validate_text, "Pack antigo: falta max_model_len=8192"
print("Pack self-check OK.", flush=True)

print("\n=== 5. Discover/download adapters ===", flush=True)
def extract_adapter_zips(root: Path):
    extracted = []
    for zip_path in root.rglob("*.zip"):
        out = zip_path.with_suffix("")
        out.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            if "adapter_config.json" in names and "adapter_model.safetensors" in names:
                zf.extract("adapter_config.json", out)
                zf.extract("adapter_model.safetensors", out)
                extracted.append(out)
                print("extraido:", zip_path, "->", out, flush=True)
    return extracted

def find_adapter_dirs(*roots: Path):
    out = []
    for root in roots:
        if not root.exists():
            continue
        for cfg in root.rglob("adapter_config.json"):
            if (cfg.parent / "adapter_model.safetensors").exists():
                out.append(cfg.parent)
    def mtime_key(path: Path):
        try:
            return ((path / "adapter_model.safetensors").stat().st_mtime, str(path))
        except Exception:
            return (0, str(path))
    return sorted(set(out), key=mtime_key, reverse=True)

# Known anchors are downloaded to local /content only. This avoids filling Drive
# with 6GB+ of calibration anchor zips/full adapters.
for name in DEFAULT_ANCHORS:
    for stale in [DRIVE_ADAPTERS_ROOT / f"{name}.zip", DRIVE_ADAPTERS_ROOT / name]:
        if stale.exists():
            if stale.is_dir():
                shutil.rmtree(stale)
            else:
                stale.unlink()
            print("anchor antigo removido do Drive:", stale, flush=True)

extract_adapter_zips(DRIVE_ADAPTERS_ROOT)

if CANDIDATE_ONLY:
    print("Modo candidate-only: pulando download/execucao dos anchors.", flush=True)
    if LOCAL_ANCHORS_ROOT.exists():
        shutil.rmtree(LOCAL_ANCHORS_ROOT)
    LOCAL_ANCHORS_ROOT.mkdir(parents=True, exist_ok=True)
    adapter_dirs = find_adapter_dirs(DRIVE_ADAPTERS_ROOT)
else:
    print("Garantindo anchors locais para calibracao...", flush=True)
    if LOCAL_ANCHORS_ROOT.exists():
        shutil.rmtree(LOCAL_ANCHORS_ROOT)
    LOCAL_ANCHORS_ROOT.mkdir(parents=True, exist_ok=True)
    for name, spec in DEFAULT_ANCHORS.items():
        dst = LOCAL_ANCHORS_ROOT / f"{name}.zip"
        downloaded = hf_hub_download(
            repo_id="felipesp1983/kg1-nemotron-training",
            repo_type="dataset",
            filename=spec["hf_file"],
            token=HF_TOKEN or None,
        )
        shutil.copy2(downloaded, dst)
        print("anchor baixado localmente:", name, "->", dst, flush=True)
    extract_adapter_zips(LOCAL_ANCHORS_ROOT)
    adapter_dirs = find_adapter_dirs(LOCAL_ANCHORS_ROOT, DRIVE_ADAPTERS_ROOT)

if CANDIDATE_FILTER:
    adapter_dirs = [p for p in adapter_dirs if CANDIDATE_FILTER in str(p)]
    print("Filtro aplicado:", CANDIDATE_FILTER, "restantes=", len(adapter_dirs), flush=True)

freed = 0
if DELETE_ANCHOR_ZIPS_AFTER_EXTRACT:
    for name in DEFAULT_ANCHORS:
        z = LOCAL_ANCHORS_ROOT / f"{name}.zip"
        extracted_model = LOCAL_ANCHORS_ROOT / name / "adapter_model.safetensors"
        if z.exists() and extracted_model.exists():
            size = z.stat().st_size
            z.unlink()
            freed += size
            print("zip removido apos extracao:", z, "freed_gb=", round(size / 1024**3, 3), flush=True)
if freed:
    print("Total liberado removendo zips de anchors:", round(freed / 1024**3, 3), "GB", flush=True)

print("Adapters encontrados:", len(adapter_dirs), flush=True)
for p in adapter_dirs[:50]:
    print("-", p, flush=True)
assert adapter_dirs, "Nenhum adapter encontrado/baixado"

print("\n=== 6. Calibration pairs / holdout ===", flush=True)
CALIBRATION_PAIRS = {}
if not CANDIDATE_ONLY:
    for name, spec in DEFAULT_ANCHORS.items():
        candidate_dir = LOCAL_ANCHORS_ROOT / name
        if (candidate_dir / "adapter_config.json").exists():
            CALIBRATION_PAIRS[str(candidate_dir)] = spec["kaggle_score"]

calibration_path = OUT_DIR / "calibration_pairs.json"
calibration_path.write_text(json.dumps(CALIBRATION_PAIRS, indent=2, sort_keys=True), encoding="utf-8")
print(calibration_path, flush=True)
print(json.dumps(CALIBRATION_PAIRS, indent=2, sort_keys=True), flush=True)
if REQUIRE_CALIBRATION_ANCHORS:
    assert len(CALIBRATION_PAIRS) >= 2, "Calibracao minima precisa dos anchors canonical_086 e v088_084"
elif not CALIBRATION_PAIRS:
    print("Sem calibracao neste run: use este resultado como triagem candidate-only.", flush=True)

DATASET_PATH = WORK_DIR / DATASET_RELATIVE
assert DATASET_PATH.exists(), DATASET_PATH
print(DATASET_PATH, "lines=", sum(1 for _ in DATASET_PATH.open(encoding="utf-8")), flush=True)

print("\n=== 7. H100 pre-score run V2 ===", flush=True)
if LOCAL_OFFLOAD_DIR.exists():
    shutil.rmtree(LOCAL_OFFLOAD_DIR)
LOCAL_OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)

adapter_root_args = [str(DRIVE_ADAPTERS_ROOT)] if CANDIDATE_ONLY else [str(LOCAL_ANCHORS_ROOT), str(DRIVE_ADAPTERS_ROOT)]
argv = [
    "--adapters-root", *adapter_root_args,
    "--dataset-path", str(DATASET_PATH),
    "--calibration-pairs", str(calibration_path),
    "--n-samples", str(N_SAMPLES),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--score-mode", SCORE_MODE,
    "--output-dir", str(OUT_DIR),
    "--offload-dir", str(LOCAL_OFFLOAD_DIR),
    "--max-adapters", str(MAX_ADAPTERS),
    "--fingerprint-mode", FINGERPRINT_MODE,
    "--gpu-memory-margin-gb", str(GPU_MEMORY_MARGIN_GB),
]
if USE_CACHE:
    argv.append("--use-cache")
run_script_in_process(batch_script, argv)

print("\n=== 8. Outputs ===", flush=True)
for path in sorted(OUT_DIR.glob("*.json")):
    print("\n###", path, flush=True)
    txt = path.read_text(encoding="utf-8", errors="replace")
    print(txt[:6000], flush=True)

print("\n=== DONE ===", flush=True)
print("Nao use score local bruto como Kaggle score. Use apenas como gate calibrado contra anchors conhecidos.", flush=True)


=== 1. GPU / Python ===

$ nvidia-smi
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

=== 2. Install dependencies ===

$ pip -q install -U "transformers==5.3.0" "peft==0.18.1" accelerate bitsandbytes safetensors huggingface_hub sentencepiece tqdm kaggle "jedi>=0.16"

$ pip -q install -U "torchao>=0.16.0"

$ pip -q install -U packaging ninja wheel setuptools

$ MAX_JOBS=4 pip -q install -U "causal-conv1d>=1.4.0" "mamba-ssm>=2.2.4" --no-build-isolation
torchao: 0.17.0
transformers: 5.3.0

=== 3. Mount Drive / secrets ===
WORK_DIR: /content/kg1_prescore
DRIVE_DIR: /content/drive/MyDrive/KG1_Nemotron_prescore
DRIVE_ADAPTERS_ROOT: /content/drive/MyDrive/KG1_Nemotron_prescore/adapters
LOCAL_ANCHORS_ROOT: /content/kg1_prescore_anchors
OUT_DIR: /content/drive/MyDrive/KG1_Nemotron_prescore/prescore_outputs_h100_v2

=== 4. Download/extract latest pre-score pack ===


prescore/kg1_prescore_calibration_pack.z(…):   0%|          | 0.00/546k [00:00<?, ?B/s]

Pack salvo em: /content/drive/MyDrive/KG1_Nemotron_prescore/kg1_prescore_calibration_pack.zip size= 545539
Pack extraido.
Pack self-check OK.

=== 5. Discover/download adapters ===
Modo candidate-only: pulando download/execucao dos anchors.
Adapters encontrados: 0


AssertionError: Nenhum adapter encontrado/baixado